# AAI-540 Data Lake — AWS Starter Notebook

Connects to the project data lake (`s3://jonno-lucas-steve-bucket/usd-aai540-group1/`)
and the Glue catalog (`aai540_silver`, `aai540_gold`) and demonstrates the four
typical query patterns:

1. Load the Gold training matrix into pandas
2. Quick EDA + Y-over-time plot
3. Ad-hoc Silver-layer query (events × dim_county)
4. Baseline OLS model — time-split on the panel

**Region:** `us-east-2` · **Workgroup:** `primary` ·
Athena results land at `s3://jonno-lucas-steve-bucket/usd-aai540-group1/athena-results/`.

## Setup

Most SageMaker images already have `awswrangler`, `boto3`, `pandas`,
`numpy`, `sklearn`, `matplotlib`, `seaborn` pre-installed. If you hit
`ImportError`, uncomment the install line below.

In [ ]:
# !pip install -q awswrangler matplotlib seaborn scikit-learn polars

In [ ]:
import boto3
import awswrangler as wr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format

# Confirm which IAM identity this notebook is running as
sts = boto3.client("sts", region_name="us-east-2")
print(sts.get_caller_identity()["Arn"])

In [ ]:
# ---- Constants ----
REGION     = "us-east-2"
BUCKET     = "jonno-lucas-steve-bucket"
PROJECT    = "usd-aai540-group1"
SILVER_DB  = "aai540_silver"
GOLD_DB    = "aai540_gold"
ATHENA_OUT = f"s3://{BUCKET}/{PROJECT}/athena-results/"

# awswrangler picks up region from the boto3 default session
boto3.setup_default_session(region_name=REGION)

## 1. Load the Gold training matrix

The full table is small (~2,500 rows × ~20 columns), so we pull it all
into memory. For Silver-layer queries (millions of rows) you'd want to
filter at the SQL level — Athena charges by data scanned.

In [ ]:
df = wr.athena.read_sql_query(
    "SELECT * FROM model_training_matrix",
    database=GOLD_DB,
    s3_output=ATHENA_OUT,
)

print(f"shape:      {df.shape}")
print(f"year range: {df['year'].min()}-{df['year'].max()}")
print(f"counties:   {df['county_fips'].nunique()}")
print(f"rows w/ events:           {(df['n_events'] > 0).sum()}")
print(f"rows w/ est_attendance:   {(df['total_est_attendance'] > 0).sum()}")
df.head()

In [ ]:
# Schema + null check
pd.DataFrame({
    "dtype":    df.dtypes,
    "n_nulls":  df.isna().sum(),
    "n_unique": df.nunique(),
})

## 2. Quick EDA

How does Y move over time? Which X features correlate with Y?
Which counties dominate the panel?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Statewide Y over time
yearly = df.groupby("year", as_index=False)["taxable_sales_usd"].sum()
yearly["billions"] = yearly["taxable_sales_usd"] / 1e9
axes[0].plot(yearly["year"], yearly["billions"], marker="o")
axes[0].set_title("CA total taxable sales by year ($B)")
axes[0].set_xlabel("year"); axes[0].set_ylabel("$ billions")

# Top-5 counties' share of Y, 2022
df_2022 = df[df["year"] == 2022].groupby("county_name")["taxable_sales_usd"].sum().sort_values(ascending=False)
top5 = df_2022.head(5)
axes[1].barh(top5.index[::-1], top5.values[::-1] / 1e9)
axes[1].set_title("Top 5 CA counties by 2022 taxable sales ($B)")
axes[1].set_xlabel("$ billions")

plt.tight_layout()
plt.show()

In [ ]:
# Pearson correlation with Y
features = [
    "n_events", "total_est_attendance", "total_expected_attendance",
    "n_festivals", "total_festival_attendance",
    "total_wages_usd", "avg_employment", "establishment_count",
    "population", "median_household_income", "median_age",
]
target = "taxable_sales_usd"

# Filter to rows where we have BOTH events AND wages (the model's training set)
train_pool = df.dropna(subset=features + [target])
print(f"training-eligible rows: {len(train_pool)}")

corr = train_pool[features + [target]].corr()[target].sort_values(ascending=False)
print("\nPearson correlation with taxable_sales_usd:")
print(corr.to_string())

## 3. Ad-hoc Silver query

`awswrangler` runs arbitrary SQL against the Glue catalog. Pull just
what you need — Athena bills by data scanned.

In [ ]:
# Example: events for the top 3 CA counties in 2022, by source
events_2022 = wr.athena.read_sql_query(
    """
    SELECT e.source,
           c.county_name,
           COUNT(*) AS n_events,
           SUM(COALESCE(e.expected_attendance, 0)) AS total_expected_attendance
    FROM aai540_silver.events e
    JOIN aai540_silver.dim_county c
      ON e.county_fips = c.county_fips
    WHERE e.year = 2022
      AND c.state_fips = '06'
      AND c.county_fips IN ('06037', '06059', '06073')  -- LA, Orange, San Diego
    GROUP BY e.source, c.county_name
    ORDER BY c.county_name, e.source
    """,
    database=SILVER_DB,
    s3_output=ATHENA_OUT,
)
events_2022

In [ ]:
# Example: venue capacity coverage — how much of our event volume has
# named-source (non-default) capacity attached?
venue_coverage = wr.athena.read_sql_query(
    """
    SELECT capacity_source,
           COUNT(DISTINCT venue_id) AS n_venues,
           SUM(n_events)            AS n_events,
           ROUND(AVG(capacity), 0)  AS avg_capacity
    FROM aai540_silver.venue_capacities
    GROUP BY capacity_source
    ORDER BY n_events DESC
    """,
    database=SILVER_DB,
    s3_output=ATHENA_OUT,
)
venue_coverage

## 4. Baseline OLS model

Time-based train/test split (NOT random) — we want to predict *future*
quarters from past ones. Train on 2015-2021, test on 2022-2023. This is
a deliberately simple OLS baseline; the real model goes in PyMC.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Drop rows missing any X (mostly the most recent quarters where QCEW lags)
mdl_df = df.dropna(subset=features + [target]).copy()
mdl_df["log_y"] = np.log1p(mdl_df["taxable_sales_usd"])

train = mdl_df[mdl_df["year"] <= 2021]
test  = mdl_df[mdl_df["year"].between(2022, 2023)]

X_train, y_train = train[features], train["log_y"]
X_test,  y_test  = test[features],  test["log_y"]

model = LinearRegression().fit(X_train, y_train)
pred  = model.predict(X_test)

print(f"train rows: {len(train):>5}   test rows: {len(test):>5}")
print(f"  test R²:  {r2_score(y_test, pred):.3f}")
print(f"  test MAE: {mean_absolute_error(y_test, pred):.3f}  (log $)")

coef_df = (
    pd.DataFrame({"feature": features, "coef": model.coef_})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
)
coef_df

In [ ]:
# Predicted vs actual on holdout
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, pred, alpha=0.4, s=15)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
ax.plot(lims, lims, "k--", lw=1)
ax.set_xlabel("Actual log(taxable_sales_usd)")
ax.set_ylabel("Predicted log(taxable_sales_usd)")
ax.set_title("OLS baseline — 2022-2023 holdout")
plt.show()

## What to try next

- **Add lagged Y as a feature.** `taxable_sales_usd` from the prior
  quarter is the strongest predictor of the current quarter.
- **County fixed effects.** One-hot encode `county_fips` — captures
  baseline economic activity unrelated to events.
- **Switch to PyMC.** `pip install pymc` and rebuild as a hierarchical
  Bayesian regression: state-level effects with county-level random
  effects. That lets you answer "for an event with X attendees in
  county Q, the expected Y impact is …" with proper uncertainty.
- **Write predictions back to S3** so the team can compare runs:
  `wr.s3.to_parquet(pred_df, f"s3://{BUCKET}/{PROJECT}/gold/predictions/run_2026_05_18.parquet")`

## See also

- `docs/PARTNER_QUICKSTART.md` — full setup + four-interface walkthrough
- `docs/DATA_SOURCES.md` — every Silver table explained, with row counts
- `sql/gold/model_training_matrix.sql` — the CTAS that builds this Gold table
- `pipelines/build_venue_capacities.py` — venue capacity reference (~1000 CA venues)